In [33]:
import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet as wn


# {
#     "date": "2024/06/03",
#     "contest": "NYT Connections 358 - June 3rd, 2024",
#     "words": [
#         "LASER",
#         "PLUCK",
#         "THREAD",
#         "WAX",
#         "COIL",
#         "SPOOL",
#         "WIND",
#         "WRAP",
#         "HONEYCOMB",
#         "ORGANISM",
#         "SOLAR PANEL",
#         "SPREADSHEET",
#         "BALL",
#         "MOVIE",
#         "SCHOOL",
#         "VITAMIN"
#     ],
#     "answers": [
#         {
#             "answerDescription": "REMOVE, AS BODY HAIR",
#             "words": [
#                 "LASER",
#                 "PLUCK",
#                 "THREAD",
#                 "WAX"
#             ]
#         },
#         {
#             "answerDescription": "TWIST AROUND",
#             "words": [
#                 "COIL",
#                 "SPOOL",
#                 "WIND",
#                 "WRAP"
#             ]
#         },
#         {
#             "answerDescription": "THINGS MADE OF CELLS",
#             "words": [
#                 "HONEYCOMB",
#                 "ORGANISM",
#                 "SOLAR PANEL",
#                 "SPREADSHEET"
#             ]
#         },
#         {
#             "answerDescription": "B-___",
#             "words": [
#                 "BALL",
#                 "MOVIE",
#                 "SCHOOL",
#                 "VITAMIN"
#             ]
#         }
#     ],
#     "difficulty": 3.3
# },

laser = wn.synset('laser.n.01')

[nltk_data] Downloading package wordnet to /home/vgupta8/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [34]:
# "LASER",
# "PLUCK",
# "THREAD",
# "WAX"

print(wn.synsets('laser')[0].definition())
len(wn.synsets('laser')[0].lemmas())

print(sorted(laser.hypernyms()))

pluck = wn.synset('pluck.v.01')
thread = wn.synset('thread.n.01')
wax = wn.synset('wax.n.01')

lpluck = laser.path_similarity(pluck)
lthread = laser.path_similarity(thread)
lwax = laser.path_similarity(wax)

print("laser-pluck:", lpluck)
print("laser-thread:", lthread)
print("laser-wax:", lwax)

pthread = pluck.path_similarity(thread)
pwax = pluck.path_similarity(wax)

print("pluck-thread:", pthread)
print("pluck-wax:", pwax)

twax = thread.path_similarity(wax)

print("thread-wax:", twax)


an acronym for light amplification by stimulated emission of radiation; an optical device that produces an intense monochromatic beam of coherent light
[Synset('optical_device.n.01')]
laser-pluck: 0.07692307692307693
laser-thread: 0.125
laser-wax: 0.07142857142857142
pluck-thread: 0.08333333333333333
pluck-wax: 0.06666666666666667
thread-wax: 0.07692307692307693


In [35]:
from nltk.corpus import wordnet_ic
nltk.download('wordnet_ic')
brown_ic = wordnet_ic.ic('ic-brown.dat')
semcor_ic = wordnet_ic.ic('ic-semcor.dat')
from nltk.corpus import genesis
nltk.download('genesis')
genesis_ic = wn.ic(genesis, False, 0.0)

# Function for GPT to call WordNet
def analyze_wordnet_relationships(words, analysis_type="similarity"):
    """
    Analyze relationships between words using WordNet.

    Args:
        words: List of words to analyze
        analysis_type: Type of analysis ("similarity", "hypernyms", "definitions")

    Returns:
        Dictionary with analysis results
    """
    results = {}

    if analysis_type == "similarity":
        # Calculate pairwise similarities
        similarities = {}
        for i, word1 in enumerate(words):
            for j, word2 in enumerate(words):
                if i < j:
                    try:
                        syn1 = wn.synsets(word1.lower())[0] if wn.synsets(word1.lower()) else None
                        syn2 = wn.synsets(word2.lower())[0] if wn.synsets(word2.lower()) else None
                        if syn1 and syn2:
                            sim = syn1.path_similarity(syn2)
                            similarities[f"{word1}-{word2}"] = round(sim, 3) if sim else 0
                    except:
                        similarities[f"{word1}-{word2}"] = 0
        results["similarities"] = similarities

    elif analysis_type == "hypernyms":
        # Get hypernyms for each word
        hypernyms = {}
        for word in words:
            try:
                syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
                if syn:
                    hypernyms[word] = [h.name().split('.')[0] for h in syn.hypernyms()[:3]]  # Top 3 hypernyms
                else:
                    hypernyms[word] = []
            except:
                hypernyms[word] = []
        results["hypernyms"] = hypernyms

    elif analysis_type == "definitions":
        # Get definitions
        definitions = {}
        for word in words:
            try:
                syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
                if syn:
                    definitions[word] = syn.definition()
                else:
                    definitions[word] = "No definition found"
            except:
                definitions[word] = "Error getting definition"
        results["definitions"] = definitions

    return results

# NYT Connections game data
game_data = {
    "words": [
        "LASER", "PLUCK", "THREAD", "WAX",
        "COIL", "SPOOL", "WIND", "WRAP",
        "HONEYCOMB", "ORGANISM", "SOLAR PANEL", "SPREADSHEET",
        "BALL", "MOVIE", "SCHOOL", "VITAMIN"
    ]
}

[nltk_data] Downloading package wordnet_ic to
[nltk_data]     /home/vgupta8/nltk_data...
[nltk_data]   Package wordnet_ic is already up-to-date!
[nltk_data] Downloading package genesis to /home/vgupta8/nltk_data...
[nltk_data]   Package genesis is already up-to-date!


In [38]:
import ipywidgets as widgets
from IPython.display import display, Markdown

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

# System prompt for NYT Connections game
messages = [
    {"role": "system", "content": """You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

Use the analyze_wordnet_relationships function to:
- Find similarities between words (analysis_type="similarity")
- Get hypernyms (broader categories) for words (analysis_type="hypernyms") 
- Get definitions of words (analysis_type="definitions")

The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

Current puzzle words: LASER, PLUCK, THREAD, WAX, COIL, SPOOL, WIND, WRAP, HONEYCOMB, ORGANISM, SOLAR PANEL, SPREADSHEET, BALL, MOVIE, SCHOOL, VITAMIN

Think step by step and use WordNet data to support your reasoning."""}
]

# Function definitions for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_wordnet_relationships",
            "description": "Analyze semantic relationships between words using WordNet",
            "parameters": {
                "type": "object",
                "properties": {
                    "words": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of words to analyze"
                    },
                    "analysis_type": {
                        "type": "string",
                        "enum": ["similarity", "hypernyms", "definitions"],
                        "description": "Type of analysis to perform"
                    }
                },
                "required": ["words", "analysis_type"]
            }
        }
    }
]

def ask_gpt(prompt):
    # Use a fresh local conversation each call, keeping only the system prompt
    local_messages = [messages[0], {"role": "user", "content": prompt}]
    
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=local_messages,
        tools=tools,
        tool_choice="auto"
    )

    if completion.choices and len(completion.choices) > 0:
        message = completion.choices[0].message
        tool_calls = getattr(message, "tool_calls", None)

        if tool_calls:
            # Append the assistant's reply (with tool_calls) to the conversation
            local_messages.append(message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if tool_name == "analyze_wordnet_relationships":
                    tool_result = analyze_wordnet_relationships(**tool_args)
                else:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}

                # tool_call_id is required to match this result to the right call
                local_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })

            followup = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=local_messages
            )
            if followup.choices:
                return followup.choices[0].message.content

        return getattr(message, "content", None)
    return str(completion)


In [39]:
# Test the WordNet integration
test_words = ["LASER", "PLUCK", "THREAD", "WAX"]
print("Testing WordNet analysis...")
result = analyze_wordnet_relationships(test_words, "similarity")
print("Similarities:", result["similarities"])

# Test GPT with WordNet
print("\nTesting GPT with WordNet grouping...")
group_prompt = (
    "Group the following 16 words into exactly 4 groups of 4 words each, "
    "and return only the groups as lists: LASER, PLUCK, THREAD, WAX, COIL, SPOOL, WIND, WRAP, "
    "HONEYCOMB, ORGANISM, SOLAR PANEL, SPREADSHEET, BALL, MOVIE, SCHOOL, VITAMIN."
)
response = ask_gpt(group_prompt)
print("GPT Response:", response)

Testing WordNet analysis...
Similarities: {'LASER-PLUCK': 0.059, 'LASER-THREAD': 0.125, 'LASER-WAX': 0.071, 'PLUCK-THREAD': 0.062, 'PLUCK-WAX': 0.062, 'THREAD-WAX': 0.077}

Testing GPT with WordNet grouping...
GPT Response: Here are the groups formed from the provided words:

1. **Group 1**: LASER, THREAD, WAX, PLUCK
2. **Group 2**: COIL, SPOOL, WIND, WRAP
3. **Group 3**: HONEYCOMB, ORGANISM, SOLAR PANEL, VITAMIN
4. **Group 4**: SPREADSHEET, BALL, MOVIE, SCHOOL
